# Notebook 2 — Baseline Model Training (6 Classification Pipelines)

This notebook instantiates and trains 6 foundational machine learning classifiers:
- **Logistic Regression**
- **Decision Tree Classifier**
- **Random Forest Classifier**
- **K-Nearest Neighbors (KNN)**
- **Support Vector Machine (SVM)**
- **XGBoost Classifier**

Training uses the SMOTE-balanced training set; evaluation on the untouched test set (real-world imbalance).

---

## BASELINE CONFIGURATION
- Train/Test split: 80/20 (stratified)
- SMOTE applied: Training set only (5,634 → 8,278 samples, 1:1 balance)
- Test set: Pristine, unbalanced (2.767:1 imbalance ratio)
- Random state: 42 (reproducible across all models)
- Evaluation metrics: Accuracy, Precision, Recall, F1-score, ROC-AUC

In [ ]:
# ============================================================================
# CELL 1: Imports & Setup
# ============================================================================
import sys
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path for utils import
sys.path.insert(0, os.path.abspath('..'))

from utils import (
    load_and_clean_base_data,
    split_features_and_target,
    preprocess_features,
    stratified_train_test_split,
    apply_smote_balancing
)

# Scikit-Learn models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# XGBoost
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️  XGBoost not installed. Install with: pip install xgboost")

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("✓ All imports successful")

In [ ]:
# ============================================================================
# CELL 2: Data Loading & Preprocessing Pipeline
# ============================================================================
print("\n" + "=" * 80)
print("DATA LOADING & PREPROCESSING PIPELINE")
print("=" * 80)

DATA_PATH = '../WA_Fn-UseC_-Telco-Customer-Churn.csv'

# Step 1: Load & clean base data
df = load_and_clean_base_data(DATA_PATH)
print(f"\n[Step 1] Loaded dataset: {df.shape[0]:,} records × {df.shape[1]} features")

# Step 2: Extract features & target
X = df.drop(columns=['Churn'])
y = df['Churn']
print(f"[Step 2] Features: {X.shape}, Target: {y.shape}")

# Step 3: Stratified train/test split (80/20)
X_train_raw, X_test_raw, y_train, y_test = stratified_train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"[Step 3] Stratified split: Train {X_train_raw.shape[0]:,} / Test {X_test_raw.shape[0]:,}")

# Step 4: Feature preprocessing (StandardScaler + Encoding)
X_train_processed, X_test_processed, feature_names = preprocess_features(
    X_train_raw, X_test_raw
)
print(f"[Step 4] Feature preprocessing: {X_train_processed.shape[1]} engineered features")

# Step 5: SMOTE balancing (training only)
X_train_balanced, y_train_balanced = apply_smote_balancing(
    X_train_processed, y_train.values, random_state=42
)
print(f"[Step 5] SMOTE applied: {X_train_processed.shape[0]:,} → {X_train_balanced.shape[0]:,} samples")

print(f"\n✓ Pipeline complete")
print(f"  Train (balanced): {X_train_balanced.shape}")
print(f"  Test (unbalanced): {X_test_processed.shape}")
print(f"  Class distribution (test): {np.bincount(y_test.astype(int))}")

In [ ]:
# ============================================================================
# CELL 3: Model Instantiation & Training
# ============================================================================
print("\n" + "=" * 80)
print("MODEL TRAINING — 6 BASELINE CLASSIFIERS")
print("=" * 80)

# Dictionary to store trained models
models = {}
training_times = {}

import time

# ────────────────────────────────────────────────────────────────────────────
# Model 1: Logistic Regression
# ────────────────────────────────────────────────────────────────────────────
print("\n[1] Logistic Regression")
start_time = time.time()
models['LogisticRegression'] = LogisticRegression(
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)
models['LogisticRegression'].fit(X_train_balanced, y_train_balanced)
training_times['LogisticRegression'] = time.time() - start_time
print(f"    ✓ Trained in {training_times['LogisticRegression']:.3f}s")

# ────────────────────────────────────────────────────────────────────────────
# Model 2: Decision Tree Classifier
# ────────────────────────────────────────────────────────────────────────────
print("[2] Decision Tree Classifier")
start_time = time.time()
models['DecisionTree'] = DecisionTreeClassifier(
    max_depth=15,
    min_samples_split=5,
    random_state=42
)
models['DecisionTree'].fit(X_train_balanced, y_train_balanced)
training_times['DecisionTree'] = time.time() - start_time
print(f"    ✓ Trained in {training_times['DecisionTree']:.3f}s")

# ────────────────────────────────────────────────────────────────────────────
# Model 3: Random Forest Classifier
# ────────────────────────────────────────────────────────────────────────────
print("[3] Random Forest Classifier")
start_time = time.time()
models['RandomForest'] = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
models['RandomForest'].fit(X_train_balanced, y_train_balanced)
training_times['RandomForest'] = time.time() - start_time
print(f"    ✓ Trained in {training_times['RandomForest']:.3f}s")

# ────────────────────────────────────────────────────────────────────────────
# Model 4: K-Nearest Neighbors (KNN)
# ────────────────────────────────────────────────────────────────────────────
print("[4] K-Nearest Neighbors (KNN)")
start_time = time.time()
models['KNN'] = KNeighborsClassifier(
    n_neighbors=5,
    n_jobs=-1
)
models['KNN'].fit(X_train_balanced, y_train_balanced)
training_times['KNN'] = time.time() - start_time
print(f"    ✓ Trained in {training_times['KNN']:.3f}s")

# ────────────────────────────────────────────────────────────────────────────
# Model 5: Support Vector Machine (SVM)
# ────────────────────────────────────────────────────────────────────────────
print("[5] Support Vector Machine (SVM)")
start_time = time.time()
models['SVM'] = SVC(
    kernel='rbf',
    C=1.0,
    random_state=42,
    probability=True  # For ROC-AUC
)
models['SVM'].fit(X_train_balanced, y_train_balanced)
training_times['SVM'] = time.time() - start_time
print(f"    ✓ Trained in {training_times['SVM']:.3f}s")

# ────────────────────────────────────────────────────────────────────────────
# Model 6: XGBoost Classifier
# ────────────────────────────────────────────────────────────────────────────
if XGBOOST_AVAILABLE:
    print("[6] XGBoost Classifier")
    start_time = time.time()
    models['XGBoost'] = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    )
    models['XGBoost'].fit(X_train_balanced, y_train_balanced)
    training_times['XGBoost'] = time.time() - start_time
    print(f"    ✓ Trained in {training_times['XGBoost']:.3f}s")
else:
    print("[6] XGBoost Classifier - SKIPPED (not installed)")

print(f"\n✓ All models trained successfully ({len(models)} models)")

In [ ]:
# ============================================================================
# CELL 4: Model Evaluation — Test Set Predictions
# ============================================================================
print("\n" + "=" * 80)
print("MODEL EVALUATION — TEST SET PERFORMANCE")
print("=" * 80)

# Dictionary to store all predictions
predictions = {}
probabilities = {}

for model_name, model in models.items():
    # Get predictions
    y_pred = model.predict(X_test_processed)
    predictions[model_name] = y_pred
    
    # Get probability predictions for ROC-AUC
    try:
        y_proba = model.predict_proba(X_test_processed)[:, 1]
        probabilities[model_name] = y_proba
    except Exception as e:
        print(f"⚠️  {model_name}: Could not get probabilities - {e}")
        probabilities[model_name] = None

print(f"✓ Predictions generated for all {len(models)} models")

In [ ]:
# ============================================================================
# CELL 5: Baseline Metrics Computation
# ============================================================================
print("\n" + "=" * 80)
print("BASELINE METRICS COMPUTATION")
print("=" * 80)

baseline_metrics = {}

for model_name, y_pred in predictions.items():
    metrics = {
        'model_name': model_name,
        'training_time_sec': training_times[model_name],
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1_score': f1_score(y_test, y_pred, zero_division=0),
    }
    
    # ROC-AUC (requires probability predictions)
    if probabilities[model_name] is not None:
        metrics['roc_auc'] = roc_auc_score(y_test, probabilities[model_name])
    else:
        metrics['roc_auc'] = np.nan
    
    baseline_metrics[model_name] = metrics

# Create a summary DataFrame
baseline_df = pd.DataFrame.from_dict(baseline_metrics, orient='index')
baseline_df = baseline_df[['model_name', 'training_time_sec', 'accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']]

print("\n✓ Baseline Metrics Summary:")
print("─" * 100)
print(baseline_df.to_string(index=False))
print("─" * 100)

In [ ]:
# ============================================================================
# CELL 6: Detailed Model Analysis & Confusion Matrices
# ============================================================================
print("\n" + "=" * 80)
print("DETAILED MODEL ANALYSIS")
print("=" * 80)

for model_name, y_pred in predictions.items():
    print(f"\n{'─' * 80}")
    print(f"Model: {model_name}")
    print(f"{'─' * 80}")
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"\nConfusion Matrix:")
    print(f"  True Negatives:  {tn:,}")
    print(f"  False Positives: {fp:,}")
    print(f"  False Negatives: {fn:,}")
    print(f"  True Positives:  {tp:,}")
    
    # Specificity (True Negative Rate)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    print(f"\nSpecificity (TNR): {specificity:.4f}")
    
    # Classification report
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn'], zero_division=0))

In [ ]:
# ============================================================================
# CELL 7: Baseline Logging Dictionary & Export
# ============================================================================
print("\n" + "=" * 80)
print("BASELINE LOGGING DICTIONARY")
print("=" * 80)

# Comprehensive baseline logging dictionary
baseline_logging = {
    'pipeline_info': {
        'dataset_name': 'Telco Customer Churn',
        'dataset_shape_original': df.shape,
        'dataset_shape_train_balanced': X_train_balanced.shape,
        'dataset_shape_test': X_test_processed.shape,
        'engineered_features': len(feature_names),
        'random_state': 42,
        'stratification': 'stratified (80/20)',
        'smote_applied': True,
        'test_set_imbalance_ratio': f"{np.bincount(y_test.astype(int))[0] / np.bincount(y_test.astype(int))[1]:.3f}:1"
    },
    'models_trained': list(baseline_metrics.keys()),
    'baseline_metrics': baseline_metrics,
    'best_model_by_metric': {
        'accuracy': baseline_df.loc[baseline_df['accuracy'].idxmax(), 'model_name'],
        'precision': baseline_df.loc[baseline_df['precision'].idxmax(), 'model_name'],
        'recall': baseline_df.loc[baseline_df['recall'].idxmax(), 'model_name'],
        'f1_score': baseline_df.loc[baseline_df['f1_score'].idxmax(), 'model_name'],
        'roc_auc': baseline_df.loc[baseline_df['roc_auc'].idxmax(), 'model_name']
    }
}

# Pretty print the logging dictionary
import json
print("\nBaseline Logging Dictionary (JSON format):\n")
print(json.dumps(baseline_logging, indent=2, default=str))

# Export to CSV for easy reference
baseline_df.to_csv('../models_baseline_metrics.csv', index=False)
print("\n✓ Baseline metrics exported to: models_baseline_metrics.csv")

# Export full logging to JSON
with open('../baseline_logging.json', 'w') as f:
    json.dump(baseline_logging, f, indent=2, default=str)
print("✓ Full baseline logging exported to: baseline_logging.json")

In [ ]:
# ============================================================================
# CELL 8: Performance Visualization
# ============================================================================
import plotly.express as px
import plotly.graph_objects as go

# Plot 1: Model comparison across metrics
fig = px.bar(
    baseline_df,
    x='model_name',
    y=['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc'],
    barmode='group',
    labels={'value': 'Score', 'model_name': 'Model'},
    title='Baseline Model Comparison — Evaluation Metrics',
    template='plotly_white',
    color_discrete_sequence=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
)
fig.update_layout(
    title=dict(x=0.5, xanchor='center', font=dict(size=18)),
    xaxis_title='Model',
    yaxis_title='Score',
    font=dict(size=11),
    height=500
)
fig.show()

# Plot 2: Training time comparison
fig_time = px.bar(
    baseline_df.sort_values('training_time_sec'),
    x='training_time_sec',
    y='model_name',
    color='training_time_sec',
    color_continuous_scale='Reds',
    labels={'training_time_sec': 'Training Time (seconds)', 'model_name': 'Model'},
    title='Training Time Comparison',
    template='plotly_white',
    orientation='h'
)
fig_time.update_layout(
    title=dict(x=0.5, xanchor='center', font=dict(size=18)),
    font=dict(size=11)
)
fig_time.show()

print("✓ Performance visualizations generated")

In [ ]:
# ============================================================================
# CELL 9: Key Findings & Recommendations
# ============================================================================
print("\n" + "=" * 80)
print("BASELINE KEY FINDINGS & RECOMMENDATIONS")
print("=" * 80)

# Identify best performer
best_model_f1 = baseline_df.loc[baseline_df['f1_score'].idxmax()]

print(f"""
📊 BASELINE SUMMARY:

✓ Total Models Trained: {len(models)}
✓ Evaluation Set: {len(y_test):,} test samples (imbalanced: 2.767:1)
✓ Training Set: {len(y_train_balanced):,} samples (SMOTE-balanced: 1:1)

🏆 BEST PERFORMING MODEL (by F1-score):
  - Model: {best_model_f1['model_name']}
  - F1-Score: {best_model_f1['f1_score']:.4f}
  - Accuracy: {best_model_f1['accuracy']:.4f}
  - Recall: {best_model_f1['recall']:.4f} (minority class detection)
  - Precision: {best_model_f1['precision']:.4f}
  - ROC-AUC: {best_model_f1['roc_auc']:.4f}

📈 METRIC INTERPRETATIONS:
  - Accuracy: Overall correctness (biased toward majority class)
  - Precision: Of predicted churners, how many actually churned?
  - Recall: Of actual churners, how many did we identify?
  - F1-Score: Harmonic mean of precision and recall (best for imbalanced)
  - ROC-AUC: Discriminative ability across all thresholds

🔍 NEXT STEPS:
  1. Hyperparameter tuning for top-3 performers
  2. Cross-validation analysis
  3. Feature importance extraction
  4. Threshold optimization for business objectives
  5. Ensemble methods (stacking, voting)
""")